&emsp;&emsp;同时，在构建索引结束后会分别生成`cache`、`logs`和`output`三个文件夹，分别存储了构建索引过程中的缓存文件、日志文件和输出文件。

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250715174838051.png" alt="image-20250715174838051" style="zoom:50%;" />

&emsp;&emsp;其中，`output`文件夹中会生成很多个`xxx.parquet`文件，这些文件就是`GraphRAG` 的索引文件，也是我们后续进行检索的依据。

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250715174658799.png" alt="image-20250715174658799" style="zoom:50%;" />

&emsp;&emsp;最后，总结一下`GraphRAG` 的索引构建流程，具体如下：
1. 配置Python运行的虚拟环境
2. 安装`Microsoft GraphRAG` 的`CLI`工具
3. 初始化配置文件
4. 配置`.env`文件
5. 配置`settings.yaml`文件
6. 准备数据文件
7. 执行索引构建

&emsp;&emsp;这个执行过程并不是很复杂，但需要大家注意的是，在执行索引构建的过程中，会涉及到多次大模型的调用，因此需要大家确保当前的网络环境是通畅的，否则可能会导致索引构建失败。同时，如果大家使用的是`ollama`启动的本地模型，那么需要确保本地模型是正常启动的，否则也会导致索引构建失败。

&emsp;&emsp;`Microsoft GraphRAG` 的索引构建流程，核心实现的是：利用大模型从传入的文档内容中提取出节点（实体）和边（关系），然后利用社区检测算法对整个知识图谱进行划分，划分成多个包含了相关性较高的节点和边的子图，然后利用大模型对每个子图进行总结，生成社区报告（摘要）（该社区报告（摘要）用来描述每个子图的概况）。当进行问答检索的时候，每个社区会依次执行`Query`的检索，最终将每个社区的结果进行汇总，从而生成全局性的完整答案。


&emsp;&emsp;自然，这样的流程究竟涉及哪些技术细节，又是如何实现的呢？以及生成的这些文件又有什么用处，该如何使用呢？

&emsp;&emsp;我们接下来就进行详细的讲解。
pip install tabulate

&emsp;&emsp;在对文档进行分块成`TextUnit`以后，接下来就要对每个`TextUnit`中的内容进行图元素的提取。 元素主要包括：实体、关系。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503071034125.png" width=80%></div>

&emsp;&emsp;主要有两种提取策略：

- 默认的提取策略是：`graph_intelligence`，其实就是编写提示词让大模型来提取出实体和关系。
- 另一种是使用`NLTK`进行实体和关系的提取。

&emsp;&emsp;这里建议大家还是选择使用大模型来进行实体和关系的提取，因为大模型提取的效果会更好。自然，这个环节需要加载我们在`settings.yml`文件中配置的对话模型接口服务。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507081304376.png" width=80%></div>

&emsp;&emsp;同时，在`settings.yml`文件中，我们还可以配置实体的类型和提示词，位置如下图所示：

&emsp;&emsp;默认的实体类型是："organization", "person", "geo", "event"

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507081438343.png" width=80%></div>

&emsp;&emsp;默认的实体类型是："organization", "person", "event", 而提示词内容就是在初始化`graphrag`项目时，在`promots`文件夹下的`extract_graph.txt`文件中定义的。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507081439713.png" width=80%></div>

&emsp;&emsp;这里的问题是：不同的数据块（chunks）可能会抽取出相同的实体。`Microsoft GraphRAG`会采用`merge`操作，如果遇到相同的节点，那么`GraphRAG`就会执行`concat`操作，也就是将对应的属性和关系进行合并。

&emsp;&emsp;同时，对节点和关系分别基于`TextUnit`的原始文本进行摘要。简单理解就是给出一个解释：为什么这个节点和关系会被提取出来。

&emsp;&emsp;这就是第三个阶段的完整流程，即从`TextUnit`中提取出实体和关系以及生成对应的摘要。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503071121796.png" width=80%></div>

&emsp;&emsp;执行`extract_graph`的`workflow`后，在`output`文件夹下会生成`entities.parquet`文件和`relations.parquet`文件，同样，这两个文件也涉及二次更新，即在提取出实体和关系后，会执行`finalize_graph`的`workflow`，分别对`entities.parquet`文件和`relations.parquet`文件进行进一步处理。在这个过程中，将执行图节点和关系的聚合，并生成图的向量表示。

In [1]:
# 请把本文件放入test文件夹下运行
from tabulate import tabulate
import pandas as pd
# 使用绝对路径读取 Parquet 文件
df_entities = pd.read_parquet('output/entities.parquet')  # 替换为实际路径
df_relations = pd.read_parquet('output/relationships.parquet')  # 替换为实际路径

In [2]:
print(tabulate(df_entities, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[10, 5, 10, 10, 25, 20, 10, 10, 10, 10]))

+------------+-------------------+------------+------------+---------------------------+----------------------+-----------+--------+-----+-----+
| id         | human_readable_id | title      | type       | description               | text_unit_ids        | frequency | degree | x   | y   |
+------------+-------------------+------------+------------+---------------------------+----------------------+-----------+--------+-----+-----+
| 44d3927e-6 | 0                 | STEVE JOBS | PERSON     | Steve Jobs was a co-      | ['978b4116e7a59cdb80 | 1         | 0      | nan | nan |
| 51f-4232-a |                   |            |            | founder of Apple Inc. and | 35ecc25d467c500cda95 |           |        |     |     |
| a0f-f9d0d5 |                   |            |            | played a pivotal role in  | eb850b8dbbb69f6d4716 |           |        |     |     |
| 28f644     |                   |            |            | its development,          | 62ab9f49362a1b80d571 |           |       

In [3]:
print(tabulate(df_relations, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[10, 5, 10, 10, 25, 20, 10, 10, 10, 10]))

+------------+-------------------+------------+------------+---------------------------+--------+-----------------+---------------+
| id         | human_readable_id | source     | target     | description               | weight | combined_degree | text_unit_ids |
+------------+-------------------+------------+------------+---------------------------+--------+-----------------+---------------+
| 5d9714f9-a | 0                 | APPLE      | GOOGLE     | Apple and Google          | 12.0   | 11              | ['37c77230    |
| 249-4022-b |                   |            |            | collaborate on certain    |        |                 | 43c9621b4a    |
| 4be-c0128b |                   |            |            | projects while competing  |        |                 | 71315295d4    |
| 1f9e34     |                   |            |            | in others, such as        |        |                 | d29b98218d    |
|            |                   |            |            | smartphones and

&emsp;&emsp;现在有了可用的实体和关系图，但是这些实体和关系都是孤立的，没有形成一个完整的图谱。因此需要做的就是将这些实体和关系进行聚合，形成一个完整的图谱。所以接下来的任务就是要将识别出来的实体和关系分组成相关关联的子集。 

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503071137479.png" width=80%></div>

&emsp;&emsp;要实现这个目标，就需要大家理解社区检测这个概念。

&emsp;&emsp;社区检索是图论中的一个重要任务，旨在识别图中节点的聚集结构。其中社区是指在图中，节点之间的连接比与其他节点的连接更为密切的子集。通过识别社区，我们就可以理解数据的内在结构，发现潜在的模式和关系。其中莱顿算法（Leiden Algorithm）是一种用于社区检测的高效算法，旨在优化社区结构的识别过程。它是基于 Louvain 算法的改进，具有更好的性能和准确性。

&emsp;&emsp;微软实现的就是分层的莱顿算法（Hierarchical Leiden Algorithm），它将图中的节点分成多个层次的社区，从而形成一个层次化的社区结构。

In [4]:
# 使用绝对路径读取 Parquet 文件
df_communities = pd.read_parquet('output/communities.parquet')  # 替换为实际路径

In [5]:
print(tabulate(df_communities, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[10, 5, 10, 10, 25, 20, 10, 10, 10, 10]))

+------------+-------------------+-----------+-------+--------+----------+-----------+------------+------------------+---------------+------------+------+
| id         | human_readable_id | community | level | parent | children | title     | entity_ids | relationship_ids | text_unit_ids | period     | size |
+------------+-------------------+-----------+-------+--------+----------+-----------+------------+------------------+---------------+------------+------+
| 94319cce-d | 0                 | 0         | 0     | -1     | []       | Community | ['8bef181d | ['12553963       | ['37c77230    | 2025-11-20 | 7    |
| 964-4398-9 |                   |           |       |        |          | 0         | -ba44-4744 | -6def-4241       | 43c9621b4a    |            |      |
| 8a3-3b5660 |                   |           |       |        |          |           | -9c20-2905 | -b6a0-cd07       | 71315295d4    |            |      |
| 822f19     |                   |           |       |        |       

&emsp;&emsp;莱顿算法（Louvain Method）通过优化模块度（modularity）来划分图中的节点，并没有固定的默认划分数量。它的目标是根据图的结构自动识别社区的数量。算法会在每个 `level` 中优化社区划分，直到达到局部最优。每个 `level` 代表一个图的状态，随着算法的迭代，图的结构会逐渐简化，节点会被聚合成更大的社区。

&emsp;&emsp;在生成社区后，执行的`workflow`是`create_final_text_units`，其实就是把之前处理的实体、关系等关联起来，形成最终的文本单元。形成一个完整的知识图谱表示，这样，每个文本单元不仅包含原始文本，还包含了与之相关的所有结构化信息（实体、关系）的引用，可以用于进一步 `Embedding` 和 后续的`Query`操作。


In [6]:
df_text_units = pd.read_parquet('output/text_units.parquet')  # 替换为实际路径


print(tabulate(df_text_units, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[20, 10, 40, 10, 20, 20]))

+----------------------+-------------------+------------------------------------------+----------+----------------------+----------------------+------------------------------------------+---------------+
| id                   | human_readable_id | text                                     | n_tokens | document_ids         | entity_ids           | relationship_ids                         | covariate_ids |
+----------------------+-------------------+------------------------------------------+----------+----------------------+----------------------+------------------------------------------+---------------+
| 978b4116e7a59cdb8035 | 1                 | 在过去的几十年中，全球科技行业经历了翻天 | 1200     | ['a2c186a49102fbffae | ['44d3927e-651f-4232 |                                          | []            |
| ecc25d467c500cda95eb |                   | 覆地的变化。以硅谷为中心的创新生态系统催 |          | c5c2453442cb5eb74fcd | -aa0f-f9d0d528f644'  |                                          |               |
| 850b8dbbb69f6d4716

&emsp;&emsp;接下来，要做的就是对每个社区中的节点、关系和摘要的定义进行总结。这样做的目的是为了方便查询，当查询时需要根据问题匹配知识库中的实体信息和关系信息时，只需要根据总结后的实体描述和关系描述就可以进行匹配了。不然遍历所有的`Description`进行匹配，效率会非常低下。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503071636612.png" width=80%></div>

&emsp;&emsp;同时，生成报告的策略也是借助提示工程 + 大模型实现，同时复用 实体、关系提取的部分子逻辑，因此，我们也可以在`setting.yml` 文件中进行相关提示词的配置更改：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507081443282.png" width=80%></div>

In [7]:
df_community_reports = pd.read_parquet('output/community_reports.parquet')  # 替换为实际路径

print(tabulate(df_community_reports, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[20, 20, 20, 20, 20,20, 10, 10, 10, 10, 10]))


+----------------------+-------------------+-----------+-------+--------+----------+------------+------------+--------------+------+--------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

&emsp;&emsp;最后，在最新版本的代码实现中，对各个阶段不在单独生成`Embedding`向量，而是在完成所有的索引构建流程且得到索引文件后，一次性的将实体、关系、社区报告等关键信息的`Embedding`向量表示生成，并在本地生成`Lancedb`存储，用于接下来的`Query`流程。

&emsp;&emsp;至此，这就是完成的`Microsoft GraphRAG`构建索引的内部细节。而关于`Query`阶段，我们将在接下来的课程中进行详细的讲解，从而构建出完整的基于`GraphRAG`的问答流程。

## 二、GraphRAG查询（Query）构建流程详解

&emsp;&emsp;在完成了`Microsoft GraphRAG`的索引构建后，`Microsoft GraphRAG`提供了一种更为直观、易用的查询方式，我们只需要输入自然语言查询，即可获得结构化的查询结果。

&emsp;&emsp;索引阶段我们利用大语言模型结合提示工程，从非结构化文本（`.txt`、`.csv`、`Json`）中提取出实体（Entities）与关系（Relationships），构建出了基础的`Knowledge Graph`，并且通过建立层次化的`community` 结构，`community` 以及`community_report` 的丰富语义，相较于传统基于`Cypher`的查询方式可以提供更多灵活性的`Query`操作，`Microsoft GraphRAG` 在项目开源之初是提供了`local` 和`global` 两种查询方式，分别对应了`local search` 和`global search`，而后在不断的迭代更新过程中，除了优化了`local search` 和`global search` 的效果，还新增了`DRIFT Search` 和`Multi Index Search` 作为扩展优化的可选项，以进一步丰富`Query`操作的多样性。

&emsp;&emsp;如下图所以，原图来源于`Microsoft GraphRAG` 的官方论文：https://arxiv.org/pdf/2404.16130

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503131632035.png" width=80%></div>

&emsp;&emsp;`Microsoft GraphRAG` 在查询阶段构建的流程，相较于构建索引阶段会更为直观，核心的具体步骤包括：

1. 接收用户的查询请求。
2. 根据查询所需的详细程度，选择合适的社区级别进行分析。
3. 在选定的社区级别进行信息检索。
4. 依据社区摘要生成初步的响应。
5. 将多个相关社区的初步响应进行整合，形成一个全面的最终答案。

&emsp;&emsp;通过学习`Microsoft GraphRAG`索引构建的源码大家应该已经能够清晰的知道，`Indexing` 过程中并不是在创建完第一层社区后就停止了，而是是分层的。也就是说，当创建第一层社区（即基础社区）后，会将这些社区视为节点，进一步构建更高层级的社区。这种方法就实现在知识图谱中可以以不同的粒度级别上组织和表示数据。比如第一层社区可以包含具体的实体或数据，而更高层级的社区则可以聚合这些基础社区，形成更广泛的概览。

&emsp;&emsp;因此最核心的`Local Search` 和`Global Search` 的实现，就是源于不同的粒度级别而构建出来用于处理不同类型问题的`Pipeline`, 其中：

- `Local Search` 是基于实体的检索。
- `Global Search` 则是基于社区的检索。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503131302122.png" width=80%></div>

&emsp;&emsp;因此接下来，我们就分别从源码层面，来详细介绍`Local Search` 和`Global Search` 的实现原理，并实际操作不同检索方式的查询操作。

| 模式             | 核心思路                  | 场景举例                   |
| -------------- | --------------------- | ---------------------- |
| **local**      | 只查询与“当前节点”邻居相关的子图     | “围绕某个主题节点的局部信息”        |
| **global**     | 全局搜索，忽略当前节点，直接在所有节点里找 | “在整个图中检索最相关的节点”        |
| **hybrid**     | 先局部查询，再全局补充           | “先看局部，再回溯全局补充背景”       |
| **contextual** | 根据当前上下文，动态选择检索区域      | “根据问题类型或语境决定局部/全局混合策略” |


### 1. Local Search 实现原理

&emsp;&emsp;首先来介绍`Local Search`， 即本地检索。

&emsp;&emsp;<font color=red> `Local Search` 即本地检索，是基于实体的检索。</font>本地搜索从相关实体开始，使用知识图谱来查找最相关的信息。例如，给定查询中的实体，使用的是连接节点的信息，通过辨识与查询相关的实体与关系，检索特定文本片段、摘要和关联性资料。


&emsp;&emsp;所以`Local Search` 本质上是基于实体的推理。特别适合回答“who”、“what”、“when” 类型的问题。

&emsp;&emsp;在`Microsoft GraphRAG` 源码中实现的内部原理如下图所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503141026335.png" width=80%></div>

&emsp;&emsp;我们可以通过`graphrag query --help` 命令来查看，如下图所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507081548539.png" width=80%></div>

&emsp;&emsp;各个字段参数的含义如下表所示：

<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>graphrag query 命令参数说明</font></p>
<div class="center">

| 参数名称                          | 类型    | 描述                                                                                     | 默认值         | 是否必需         |
|-----------------------------------|---------|------------------------------------------------------------------------------------------|----------------|------------------|
| `--method`                        | Tpye| 可以选择local、global、drift或basic算法。                                                                          | None           | 是               |
| `--query`                         | TEXT    | 要执行的查询，即提出的问题。                                                                          | None           | 是               |
| `--config`                        | PATH    | 要使用的配置文件路径。                                                                  | None           | 否               |
| `--data`                          | PATH    | 索引管道输出目录（即包含 parquet 文件的目录）。                                         | None           | 否               |
| `--root`                          | PATH    | 项目根目录的路径。                                                                      | .              | 否               |
| `--community-level`               | INTEGER | 从中加载社区报告的 Leiden 社区层级。较高的值表示来自较小社区的报告。                     | 2              | 否               |
| `--dynamic-community-selection`    |         | 使用动态社区选择的全局搜索。                                                            | no-dynamic-community-selection | 否               |
| `--response-type`                 | TEXT    | 描述响应类型和格式的自由文本，可以是任何内容，例如多个段落、单个段落、单句、3-7点列表、单页、多页报告。 | Multiple Paragraphs | 否               |
| `--streaming`                     |         | 以流式方式打印响应。                                                                    | no-streaming    | 否               |
| `--help`                          |         | 显示帮助信息并退出。                                                                    |                | 否               |

</div>

&emsp;&emsp;其中，在执行查询时必须指定的参数是`--method`和`--query`，其他参数为可选参数。其中：

- `--method` 参数可以选择`local`、`global`、`drift`或`basic`算法。(接下来我们会依次介绍这几种算法)
- `--query` 参数是要执行的查询，即提出的问题。

&emsp;&emsp;了解到这里，就可以通过命令行快速启动问答检索了。这里我们先来看`local` 本地搜索。输入如下命令：

```bash
    graphrag query --root ./ --method local --query "苹果公司都有哪些产品？"
```

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250715175213316.png" alt="image-20250715175213316" style="zoom:50%;" />

&emsp;&emsp;最终会显示`SUCCESS: Local Search Response` 成功提示，并会显示最终的问答结果。整个过程使用起来非常简单，但是简单并不意味着可以直接使用，大多数情况下基于通用流程的问答检索，并不能满足实际业务需求。比如检索的效果不准确，效率不高，检索结果不全面等，因此，我们需要进一步掌握`Microsoft GraphRAG` 的检索原理，并根据实际业务需求，进行针对性的优化和调整。

&emsp;&emsp;而对于 `Local Search` 的完整实现过程，其实是这样的：

1. 依次读取`text_units.parent`、`entities.parent`、`relationships.parent` `communities.parent` 和`community_reports.parent` 的索引文件，并将其加载到内存中。
2. 加载`Lancedb` 中的词向量，准备用于后续的相似度计算。
3. 根据社区的`community_level` 参数的值，对实体进行第一轮的过滤，过滤的规则是：如果实体的`community_level` 小于等于`community_level` 参数的值，则保留该实体，否则丢弃。
4. 基于输入的问题，进行实体的匹配，并构建完整的上下文。
   - 处理输入问题，如果存在对话历史记录，则将之前的用户问题附加到当前查询。
   - 将输入的问题转化为词向量，然后和lancedb中的实体词向量进行相似度计算，得到与查询最相关的实体，这个过程中会采用两个策略：
      - 过采样 (Oversampling) 策略，即最终检索的实体数量是 k * oversample_scaler。
      - 如果提供了exclude_entity_names列表，则过滤掉这些实体。
   - 根据匹配到的实体，读取该实体所属的社区报告，这个过程会采用的策略是：
    - 统计每个社区被多少个选中实体引用（一个实体可能属于多个社区），做基于实体归属的社区投票排序
    - 按匹配度和社区自身排名双重排序
        - 主要排序标准：被实体引用的频次（匹配度）
        - 次要排序标准：社区自身的重要性排名
   - 在 2,3 的基础上，提取出文本单元、关系的附件属性
   - 生成完整的数据表格
5. 构建本地搜索的系统提示词，将数据表格填充到系统提示词中，引导大模型生成最终的回答。其提示词设置在`settings.yaml` 文件的`local_search` 中

&emsp;&emsp;以上就是`Local Search` 的完整实现过程，至此，大家再理解`Microsoft GraphRAG` 给出的`Local Search` 原理图，就可以非常清晰的理解到其中各个阶段的工作原理了。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503141408842.png" width=80%></div>

&emsp;&emsp;好，`Local Search` 的源码解析到这里就全部结束了。所以的细节均给大家介绍完毕。接下来我们在关注`Global Search` 的实现过程。

### 2. Global Search 实现原理

&emsp;&emsp;`Microsoft GraphRAG` 中的全局搜索算法旨在回答需要了解整个数据集的抽象问题，即借助社区摘要来获取全局的答案。实现思路是通过 map-reduce 流程总结知识图谱中的社群摘要，汇总社区摘要中的见解，尝试生成文档中元素的概述，聚合相关资料并生成针对整体数据集的高层次回答。因此全局搜索更侧重于为需要更高层次理解的问题提供答案。比如`数据中的前5个主题是什么？`这类问题。


&emsp;&emsp;传统的 `Map-Reduce` 中，`Map`阶段会将输入数据分解为独立的块，每个块可以并行处理，然后`Reduce` 阶段会汇总所有映射操作的结果，生成最终输出。而在`Microsoft GraphRAG` 中，`Map` 阶段会使用大模型对多个文档或信息片段并行处理，从每个片段中提取相关信息，然后`Reduce` 阶段会汇总所有映射操作的结果，生成最终输出。

&emsp;&emsp;`Global Search` 在`Microsoft GraphRAG` 源码中的实现原理图如下所示：


<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503141417825.png" width=80%></div>

&emsp;&emsp;注意，当使用`Global Search` 时，需要指定`--method global` 参数，如下所示：

```bash
graphrag query --root ./ --method global --query "文本库的内容可以分为哪几个主题？"
```

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250715175521079.png" alt="image-20250715175521079" style="zoom:50%;" />

&emsp;&emsp;如果大家理解了`Local Search` 的检索过程，那么`Global Search` 理解起来就非常简单了。按照相同的思路，逐步进入到各个阶段的内部细节即可清晰的了解其实现的原理。接下来我们就分步骤给大家介绍`Global Search` 的实现过程。具体如下：

1. 依次读取`entities.parent`、`communities.parent` 和`community_reports.parent` 的索引文件，并将其加载到内存中；
2. 依次创建 `entities.parent`、`communities.parent` 和`community_reports.parent` 的实体对象，并进行格式化处理;
3. 进入到构建上下文阶段。在这个阶段，最关键的一个核心概念是：静态与动态全局搜索策略的选择。
    - 静态策略方法指的是知识图谱中预定级别的社区中进行搜索来生成答案。然后，大模型合并并总结此抽象级别的所有社区报告。最后，摘要用作 大模型的附加上下文，以生成对用户问题的响应。此为静态方法。它存在的问题是既昂贵又低效，因为包含许多对用户查询没有帮助的低级报告。
    - 动态社区选择算法`dynamic_community_selection`。该算法利用索引数据集的知识图谱结构。从知识图谱的根开始，我们使用`提示工程 + 大模型`来评估社区报告在回答用户问题方面的相关性。如果报告被视为不相关，则将其及其节点（或子社区）从搜索过程中删除。另一方面，如果报告被视为相关，将遍历其子节点并重复该操作。最后，只有相关的报告才会传递给 `map-reduce` 操作以生成对用户的响应。如下图所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503141445707.png" width=80%></div>

&emsp;&emsp;该算法类实现的核心机制并不是简单地评估所有社区，而是采用启发式遍历，具体体现在：

1. **选择性探索**: 并不是简单地评估所有社区，而是根据当前社区的相关性决定是否探索其子社区。只有当一个社区的评分大于或等于阈值时，才会将其子社区添加到下一轮评估中。
2. **动态队列构建**: queue = communities_to_rate 表明下一轮要评估的社区完全取决于当前轮次中哪些社区被认为是相关的。这不是一个固定的或预先确定的遍历顺序。
3. **剪枝机制**: 如果一个社区的评分低于阈值，其所有子社区都会被"剪枝"，不会被进一步探索。这是启发式算法的典型特征。
4. **自适应性**: 算法的路径会根据不同的查询而变化，因为相关性评分依赖于具体的查询内容。
5. **回退策略**: 如果在当前路径上找不到相关社区，算法会尝试探索下一个层级的所有社区，这也是一种启发式决策。

&emsp;&emsp;其中，用于评估社区相关性的提示词是这样的，其对应的中文提示如下所示：

```json
    Rate_query = "“”
    ——角色
    你是一个乐于助人的助手，负责决定所提供的信息是否有助于回答给定的问题，即使它只是部分相关。
    ——目标
    在0到5的范围内，请对回答问题所提供的信息的相关性或帮助程度进行评分。
    ——信息
    {描述}
    ——问题
    {问题}
    ——目标回复长度和格式——
    请以以下JSON格式回复，包含两个条目：
    -“原因”：评分的原因，请包括你考虑过的信息。
    -“评级”：相关度从0到5，其中0是最不相关的，5是最相关的。
    {{
    “理由”:str,
    “等级”:int。
    }}
```


&emsp;&emsp;其响应的结果如下所示：


<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503141605641.png" width=80%></div>

&emsp;&emsp;然后，该输出的结果会作为变量`context_data` 传递给`global_search_reduce_system_prompt.txt` 中定义的提示，并调用大模型生成最终的 `Reduce` 响应，同时，`Reduce` 响应的结果会作为变量`context_text` 传递给`global_search_knowledge_system_prompt.txt` 中定义的提示，引导大模型生成最终的 `Knowledge` 响应。

&emsp;&emsp;至此，`Global Search` 的实现原理就全部介绍完毕了。而现在大家在看`Microsoft GraphRAG` 官方提供的流程图，自然也能够非常清晰的理解其背后的实现原理。如下图所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503141625872.png" width=80%></div>

&emsp;&emsp;以上就是我们对`Microsoft GraphRAG` 的检索流程中最关键的`Local Search` 和 `Global Search`的完整流程解读，而关于`basic search`和`drift search`，大家可以按照相同的思路进行扩展学习。

## 三、GraphRAG Python API使用方法

In [1]:
from pathlib import Path
from pprint import pprint

import pandas as pd

import graphrag.api as api
from graphrag.config.load_config import load_config
from graphrag.index.typing.pipeline_run_result import PipelineRunResult

- Index过程（大约6分钟）

In [2]:
PROJECT_DIRECTORY = "/root/autodl-tmp/graphrag/test/graphrag_course"

In [3]:
graphrag_config = load_config(Path(PROJECT_DIRECTORY))

In [4]:
index_result: list[PipelineRunResult] = await api.build_index(config=graphrag_config)

# index_result is a list of workflows that make up the indexing pipeline that was run
for workflow_result in index_result:
    status = f"error\n{workflow_result.errors}" if workflow_result.errors else "success"
    print(f"Workflow Name: {workflow_result.workflow}\tStatus: {status}")

[2025-11-20T07:16:53Z WARN  lance::dataset::write::insert] No existing dataset at /root/autodl-tmp/graphrag/test/graphrag_course/output/lancedb/default-entity-description.lance, it will be created
[2025-11-20T07:16:54Z WARN  lance::dataset::write::insert] No existing dataset at /root/autodl-tmp/graphrag/test/graphrag_course/output/lancedb/default-text_unit-text.lance, it will be created


Workflow Name: create_base_text_units	Status: success
Workflow Name: create_final_documents	Status: success
Workflow Name: extract_graph	Status: success
Workflow Name: finalize_graph	Status: success
Workflow Name: create_communities	Status: success
Workflow Name: create_final_text_units	Status: success
Workflow Name: create_community_reports	Status: success
Workflow Name: generate_text_embeddings	Status: success


[2025-11-20T07:16:54Z WARN  lance::dataset::write::insert] No existing dataset at /root/autodl-tmp/graphrag/test/graphrag_course/output/lancedb/default-community-full_content.lance, it will be created


- Query过程

In [5]:
entities = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/entities.parquet")
communities = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/communities.parquet")
community_reports = pd.read_parquet(
    f"{PROJECT_DIRECTORY}/output/community_reports.parquet"
)

response, context = await api.global_search(
    config=graphrag_config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    community_level=1,
    dynamic_community_selection=False,
    response_type="Multiple Paragraphs",
    query="请帮我介绍下亚马逊公司？",
)

In [6]:
print(response)

<think>
好的，我现在需要帮助用户了解亚马逊公司。首先，我应该从分析师的报告中提取关键信息，并按照重要性排序。

第一位分析师提到亚马逊是电子商务和科技巨头，收购了Whole Foods扩展零售业务，并开发了Alexa作为智能助手平台。此外，AWS在2006年推出，成为最赚钱的业务之一。这些信息非常重要，所以我会放在前面。

接下来，第二位分析师强调亚马逊的战略扩展，包括零售、技术和云 computing。收购Whole Foods加强了零售地位，Alexa巩固了智能家居领导地位，而AWS的成功使亚马逊成为云计算领导者。这部分内容与第一位分析师的信息互补，可以合并到同一段落中。

第三位分析师提到亚马逊与其他科技巨头如Google和Apple在人工智能领域的合作，特别是通过Alexa平台推动AI技术发展。这为亚马逊的技术合作提供了额外信息，应该作为单独的一点来介绍。

现在，我需要将这些信息组织成多个段落，每个段落涵盖一个主要方面：公司概述、战略扩展、技术创新和行业影响。同时，要确保引用正确的数据记录，并避免列出超过5个ID，使用“+more”表示更多记录。

最后，检查是否有遗漏的信息或重复的内容，确保回答全面且结构清晰。
</think>

亚马逊（Amazon）是一家全球领先的电子商务和科技公司，业务涵盖零售、云计算、人工智能等多个领域。以下是对其核心业务和发展战略的详细介绍：

### 1. **核心业务与战略扩展**
亚马逊通过收购Whole Foods加强了其在零售领域的布局，进一步巩固了其作为主要零售商的地位 [Data: Reports (0, +more)]。此外，公司开发了Alexa智能助手平台，推动了智能家居技术的发展，并在此领域确立了领导地位 [Data: Reports (0, 1, +more)]。

### 2. **技术创新与市场领导**
亚马逊于2006年推出了Amazon Web Services（AWS），这一云计算服务迅速成为公司的主要收入来源之一，并使其在云计算市场中占据了领先地位 [Data: Reports (0, +more)]。AWS的成功不仅推动了公司整体增长，也为其他企业提供了强大的云基础设施支持。

### 3. **人工智能与合作**
亚马逊在人工智能领域进行了多项创新，包括Alexa平台的开发。此外，

> 需要注意的是，以上是GraphRAG高层API，还有底层的面向开发端的API。